# DREAMT Triple-Stream Sleep Staging — Colab Training

**Before you start:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Make sure your Google Drive has either:
   - `MyDrive/DREAMT/processed/` — pre-extracted numpy files (fastest, recommended)
   - `MyDrive/DREAMT/data_64Hz/` — raw subject CSVs (preprocessing runs in Colab)

Run cells top to bottom. Each cell is self-contained.

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Cell 3: Clone repository and install dependencies ─────────────────────
# Option A: clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USERNAME/sleep-staging-models.git
# %cd sleep-staging-models

# Option B: the repo is already on Drive
# %cd /content/drive/MyDrive/sleep-staging-models

# Option C: code is uploaded as a zip
# !unzip /content/drive/MyDrive/sleep-staging-models.zip -d /content/
# %cd /content/sleep-staging-models

# ── Uncomment and edit ONE option above, then run this cell ───────────────

# Install missing packages (most are pre-installed on Colab)
!pip install -q scikit-learn tqdm pyyaml seaborn

In [ ]:
# ── Cell 4: Check preprocessed files OR run preprocessing ────────────────
import os
from pathlib import Path

DRIVE_PROCESSED = '/content/drive/MyDrive/DREAMT/processed'
LOCAL_PROCESSED = '/content/dreamt_processed'   # fast local disk
DRIVE_RAWDATA   = '/content/drive/MyDrive/DREAMT/data_64Hz'

REQUIRED = ['bvp.npy', 'acc.npy', 'ibi.npy', 'labels.npy',
            'subjects.npy', 'split_boundaries.json',
            'ibi_stats.json', 'metadata.json']

def check_ready(d):
    return all((Path(d) / f).exists() for f in REQUIRED)

if check_ready(LOCAL_PROCESSED):
    PREPROCESSED_DIR = LOCAL_PROCESSED
    print(f'Using local preprocessed data: {PREPROCESSED_DIR}')

elif check_ready(DRIVE_PROCESSED):
    # Copy from Drive to local disk for much faster I/O during training
    print('Copying preprocessed data from Drive to local disk (~3 GB, ~2 min)...')
    os.makedirs(LOCAL_PROCESSED, exist_ok=True)
    !cp -r {DRIVE_PROCESSED}/. {LOCAL_PROCESSED}/
    PREPROCESSED_DIR = LOCAL_PROCESSED
    print(f'Done. Using: {PREPROCESSED_DIR}')

elif Path(DRIVE_RAWDATA).exists():
    print('Preprocessed files not found. Running preprocessing on raw CSVs...')
    print('This takes ~25 minutes.')
    !python preprocess_dreamt.py \
        --data_dir {DRIVE_RAWDATA} \
        --output_dir {LOCAL_PROCESSED}
    # Also save to Drive for future runs
    os.makedirs(DRIVE_PROCESSED, exist_ok=True)
    !cp -r {LOCAL_PROCESSED}/. {DRIVE_PROCESSED}/
    print('Saved preprocessed files to Drive for future runs.')
    PREPROCESSED_DIR = LOCAL_PROCESSED

else:
    raise RuntimeError(
        f'No data found.\n'
        f'  Expected preprocessed files at: {DRIVE_PROCESSED}\n'
        f'  Or raw CSVs at:                 {DRIVE_RAWDATA}\n'
        f'  See COLAB_QUICKSTART.md for upload instructions.'
    )

# Verify
print('\nFiles present:')
for f in REQUIRED:
    p = Path(PREPROCESSED_DIR) / f
    size = f'{p.stat().st_size/1e6:.0f} MB' if p.exists() else 'MISSING'
    print(f'  {f:35s} {size}')

In [ ]:
# ── Cell 5: Write config for this Colab run ───────────────────────────────
import yaml

colab_config = {
    'data': {
        'preprocessed_dir': PREPROCESSED_DIR,
        'data_dir': '',          # unused (numpy backend active)
        'fs': 64.0,
        'window_sec': 30.0,
        'train_ratio': 0.70,
        'val_ratio': 0.15,
        'test_ratio': 0.15,
        'seed': 42,
        'min_windows_per_subject': 10,
        'num_workers': 2,        # 2 workers is fine on Colab GPU
    },
    'model': {
        'n_classes': 6,
        'd_model': 256,
        'n_heads': 8,
        'n_fusion_blocks': 3,
        'dropout': 0.2,
        'ibi_n_features': 5,
    },
    'training': {
        'batch_size': 64,        # larger batch = faster on T4
        'num_epochs': 50,
        'patience': 15,
        'learning_rate': 1e-4,
        'weight_decay': 1e-5,
        'staged_training': False,
        'modality_dropout_prob': 0.0,
    },
    'output': {
        'save_dir': '/content/drive/MyDrive/DREAMT/outputs',
        'save_frequency': 5,
        'log_frequency': 50,
    },
    'hardware': {
        'use_amp': True,
        'gradient_accumulation_steps': 1,
        'device': 'cuda',
    },
}

config_path = '/content/config_colab.yaml'
with open(config_path, 'w') as f:
    yaml.dump(colab_config, f, default_flow_style=False)

print('Config written to', config_path)
print(yaml.dump(colab_config, default_flow_style=False))

In [ ]:
# ── Cell 6: Quick sanity check — one batch through model ──────────────────
import sys, torch
sys.path.insert(0, '.')   # ensure local modules are found

from dreamt_numpy_dataset import DreamtNumpyDataset, DatasetConfig, get_dataloaders

cfg = DatasetConfig(preprocessed_dir=PREPROCESSED_DIR)
train_ds = DreamtNumpyDataset(PREPROCESSED_DIR, split='train')

bvp, acc, ibi, label = train_ds[0]
print(f'BVP  shape: {bvp.shape}  range [{bvp.min():.2f}, {bvp.max():.2f}]  (z-scored)')
print(f'ACC  shape: {acc.shape}  range [{acc.min():.2f}, {acc.max():.2f}]  (z-scored)')
print(f'IBI  shape: {ibi.shape}  values: {ibi.numpy().round(3)}')
print(f'Label: {label}')

from triple_stream_model import create_triple_stream_model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_triple_stream_model().to(device)
model.eval()

with torch.no_grad():
    logits = model(
        bvp.unsqueeze(0).to(device),
        acc.unsqueeze(0).to(device),
        ibi.unsqueeze(0).to(device),
    )
print(f'Logits: {logits.shape}  {logits.cpu().numpy().round(3)}')
print('Sanity check passed.')

In [ ]:
# ── Cell 7: Run training ──────────────────────────────────────────────────
!python train_triple_stream.py --config /content/config_colab.yaml

In [ ]:
# ── Cell 8: (Optional) Monitor RAM and VRAM usage ─────────────────────────
import psutil, torch

ram = psutil.virtual_memory()
print(f'RAM used:  {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB  ({ram.percent:.1f}%)')

if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated() / 1e9
    reserv = torch.cuda.memory_reserved()  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used: {alloc:.2f} GB allocated, {reserv:.2f} GB reserved / {total:.1f} GB total')

In [ ]:
# ── Cell 9: (Optional) Launch TensorBoard ─────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/DREAMT/outputs